# Optimizer sandbox

Play with a real `DataPackage` outside of Frappe/bench: tweak employees, FTE targets, rules and weights, re-solve, and inspect the result. No live site needed once you have a snapshot.

**1. Capture a snapshot** from an existing Optimizer Run on `development.localhost` (any status — only its `date`/`mode`/`ruleset`/leave-speculations/existing-assignments mode are used):

```bash
bench --site development.localhost capture-datapackage --run <run-name>
```

This writes `sandbox/snapshots/<run-name>.json`. Snapshots contain real employee/leave data from your dev site, so `sandbox/snapshots/` is gitignored — re-capture instead of committing one.

**2. Pick a kernel**: the app's own Python env (`uv run python` from `apps/autoshift`, or the bench env — both already have `pulp`, `pandas`, and `autoshift` importable; see `pyproject.toml`'s `dev` dependency group for `ipykernel`/`pandas`).

In [1]:
import logging
from pathlib import Path

import pulp
from helpers import (  # ty:ignore[unresolved-import]
	assignment_frame,
	objective_breakdown,
	replace,
	room_utilization_frame,
	solve,
	status,
)

from autoshift.optimizer.rules import BUILTIN_RULES
from autoshift.optimizer.types import DataPackage

logging.basicConfig(level=logging.DEBUG)
logging.getLogger().setLevel(logging.DEBUG)

## Load a snapshot

Point this at whichever file `capture-datapackage` produced.

In [2]:
SNAPSHOT = Path("snapshots/AS-2026-06-22-051.json")

data = DataPackage.loads(SNAPSHOT.read_text())
active_rule_count = len(data.rules) or len(BUILTIN_RULES)
print(
	f"{len(data.employees)} employees, {len(data.shift_types)} shift types, "
	f"{len(data.working_days)} days ({data.working_days[0]}..{data.working_days[-1]}), "
	f"{len(data.branches)} branches, {active_rule_count} rules"
)

54 employees, 2 shift types, 10 days (2026-06-22..2026-07-03), 1 branches, 8 rules


## Solve as captured

In [3]:
prob, x, active_rooms, logs = solve(data)
print(status(prob), pulp.value(prob.objective))

DEBUG:pulp.apis.core:cbc /tmp/b5da6c87aabe43848714281e15d2388b-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/b5da6c87aabe43848714281e15d2388b-pulp.sol 


Optimal 60.0


## Inspect the solution

In [4]:
assignment_frame(data, x)

,employee,shift_type,date,branch,forced,ass
0,103,Omni AM,2026-06-22,Balexert,False,1.0
1,138,Omni AM,2026-06-22,Balexert,False,1.0
2,181,Omni AM,2026-06-22,Balexert,False,1.0
3,23,Omni AM,2026-06-22,Balexert,False,1.0
4,47,Omni AM,2026-06-22,Balexert,False,1.0
...,...,...,...,...,...,...
115,120,Omni PM,2026-07-03,Balexert,False,1.0
116,138,Omni PM,2026-07-03,Balexert,False,1.0
117,177,Omni PM,2026-07-03,Balexert,False,1.0
118,47,Omni PM,2026-07-03,Balexert,False,1.0


In [5]:
room_utilization_frame(data, active_rooms)

,discipline,shift_type,date,branch,staffed,capacity
0,Omni - CMB&B,Omni AM,2026-06-22,Balexert,6,6
1,Omni - CMB&B,Omni PM,2026-06-22,Balexert,6,6
2,Omni - CMB&B,Omni AM,2026-06-23,Balexert,6,6
3,Omni - CMB&B,Omni PM,2026-06-23,Balexert,6,6
4,Omni - CMB&B,Omni AM,2026-06-24,Balexert,6,6
5,Omni - CMB&B,Omni PM,2026-06-24,Balexert,6,6
6,Omni - CMB&B,Omni AM,2026-06-25,Balexert,6,6
7,Omni - CMB&B,Omni PM,2026-06-25,Balexert,6,6
8,Omni - CMB&B,Omni AM,2026-06-26,Balexert,6,6
9,Omni - CMB&B,Omni PM,2026-06-26,Balexert,6,6


In [6]:
# per-rule contribution to the objective (built-in objective rules only — see helpers.py)
objective_breakdown(data, x, active_rooms)

{'Objective: Room utilization': 120.0, 'Objective: Shift preferences': -60.0}

## Play with variables

`DataPackage` is frozen — use `replace(data, field=...)` (shorthand for `dataclasses.replace`) to build a variant, then re-solve and compare.

In [7]:
variant = data  # start here, then override fields above
prob2, x2, active_rooms2, _ = built = solve(variant, solve=False)

In [8]:
for var in prob2.variables():
	var.cat = pulp.LpContinuous

In [9]:
print(status(prob2), pulp.value(prob2.objective))
prob2, x2, active_rooms2, _ = built = solve(variant, solve=built)
print(status(prob2), pulp.value(prob2.objective))
assignment_frame(variant, x2)

DEBUG:pulp.apis.core:cbc /tmp/76b35c6874224560b74b427e55d2dd7f-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/76b35c6874224560b74b427e55d2dd7f-pulp.sol 


Not Solved None
Optimal 60.0


,employee,shift_type,date,branch,forced,ass
0,103,Omni AM,2026-06-22,Balexert,False,1.0
1,138,Omni AM,2026-06-22,Balexert,False,1.0
2,181,Omni AM,2026-06-22,Balexert,False,1.0
3,23,Omni AM,2026-06-22,Balexert,False,1.0
4,47,Omni AM,2026-06-22,Balexert,False,1.0
...,...,...,...,...,...,...
115,120,Omni PM,2026-07-03,Balexert,False,1.0
116,138,Omni PM,2026-07-03,Balexert,False,1.0
117,177,Omni PM,2026-07-03,Balexert,False,1.0
118,47,Omni PM,2026-07-03,Balexert,False,1.0


In [10]:
print("wat")

wat


In [11]:
# example: give one employee a bigger FTE target
# employee = data.employees[0]
# variant = replace(data, target_shifts={**data.target_shifts, employee: 40})

variant = data  # start here, then override fields above
prob2, x2, active_rooms2, _ = solve(variant)
print(status(prob2), pulp.value(prob2.objective))
assignment_frame(variant, x2)

DEBUG:pulp.apis.core:cbc /tmp/87f9591901154bdaa8bb57dda2f69004-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/87f9591901154bdaa8bb57dda2f69004-pulp.sol 


Optimal 60.0


,employee,shift_type,date,branch,forced,ass
0,103,Omni AM,2026-06-22,Balexert,False,1.0
1,138,Omni AM,2026-06-22,Balexert,False,1.0
2,181,Omni AM,2026-06-22,Balexert,False,1.0
3,23,Omni AM,2026-06-22,Balexert,False,1.0
4,47,Omni AM,2026-06-22,Balexert,False,1.0
...,...,...,...,...,...,...
115,120,Omni PM,2026-07-03,Balexert,False,1.0
116,138,Omni PM,2026-07-03,Balexert,False,1.0
117,177,Omni PM,2026-07-03,Balexert,False,1.0
118,47,Omni PM,2026-07-03,Balexert,False,1.0


## Tweak rule weights / selection

`data.rules` is a tuple of `(rule_name, builtin_key, custom_code, weight)`. An empty tuple means "every built-in rule at weight 1.0". Rebuild it to change weights, drop constraint rules (careful — most exist for correctness, not just "nice to have"), or add a rule you're drafting in `autoshift/optimizer/rule_scratchpad.py`.

In [12]:
# # example: double the room-utilization objective's weight
# base_rules = data.rules or tuple((rule.title, k, "", 1.0) for k, rule in BUILTIN_RULES.items())
# reweighted = replace(
# 	data,
# 	rules=tuple(
# 		(name, key, code, weight * 2 if key == "room_utilization_objective" else weight)
# 		for name, key, code, weight in base_rules
# 	),
# )
# prob3, x3, active_rooms3, _ = solve(reweighted)
# objective_breakdown(reweighted, x3, active_rooms3)

In [16]:
# example: double the room-utilization objective's weight
import itertools

from autoshift.optimizer.rules import RuleContext

base_rules = data.rules or tuple((rule.title, k, "", 1.0) for k, rule in BUILTIN_RULES.items())


def weigh_assignments_objective(ctx: RuleContext) -> None:
	data = ctx.data
	epsilon = 2**-10
	ctx.add_objective(
		pulp.lpSum((0 if comb in data.forced else -epsilon) * var for comb, var in ctx.x.items())
	)


new_rule = (
	"Objective: Conserve Existing Assignments",
	"",
	"""
def apply(ctx: RuleContext) -> None:
	data = ctx.data
	epsilon = 2**-10
	ctx.add_objective(
		pulp.lpSum((0 if comb in data.forced else -epsilon) * var for comb, var in ctx.x.items())
	)
	""",
	1.0,
)
reweighted = replace(
	data,
	rules=[*base_rules, new_rule],
)
prob3, x3, active_rooms3, _ = solve(reweighted)
objective_breakdown(reweighted, x3, active_rooms3)

DEBUG:pulp.apis.core:cbc /tmp/5344fee915a64a2c9061592017021e6c-pulp.mps -max -sec 30 -timeMode elapsed -solve -printingOptions all -solution /tmp/5344fee915a64a2c9061592017021e6c-pulp.sol 


{'Objective: Room utilization': 120.0,
 'Objective: Shift preferences': -60.0,
 'Objective: Conserve Existing Assignments': -0.1171875}

In [18]:
df = assignment_frame(reweighted, x3)

In [19]:
df.nunique()

employee      14
shift_type     2
date          10
branch         1
forced         1
ass            1
dtype: int64